In [0]:
%sql
select max(ingestion_date) from com_raw.kom_medical_events;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
%sql 
-- =============================================================================
-- Patient HCP Visit Summary View - Top 5 based on 3-Year activity
-- Purpose:
--   Build a patient-level summary for MPS II (E761/E763) patients including:
--   - eligibility logic (Dx criteria + evidence of treatment)
--   - HCP attribution (first Dx, first Tx, latest claim, latest Tx, most-seen top 5)
--   - derived treatment timing metrics (dx->tx months, tx period months)
--   - Tivi fill counts (windowed)
--
-- Key change implemented:
--   first_tx_after_diagnosis is ALL-TIME tx (no date filters) but must be >= incidence_date,
--   using all_tx_claims_alltime.
-- =============================================================================

CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
/* ============================================================================
   1) ELIGIBILITY COHORT BUILD
   Goal: Identify eligible MPS II patients using:
     A) "Specified" Dx (E761) with >=2 distinct Dx dates + ANY qualifying treatment evidence
     B) "Incremental Unspecified" Dx (E763) with >=2 distinct Dx dates + Tivi-only evidence
        and NOT already included in (A)
   ========================================================================== */

-- Pull all "Specified" diagnosis events (E761) within 5-year-ish window for Dx counting.
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Specified".
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Pull all "Unspecified" diagnosis events (E763) within the same window for Dx counting.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Unspecified".
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Treatment evidence universe (broad): Tivi NDCs OR relevant infusion/procedure codes.
-- Used to ensure "Specified" cohort has some treatment evidence in the more recent window.
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Treatment evidence (narrow): Tivi only (NDCs + J1743).
-- Used for incremental inclusion of "Unspecified" cohort.
MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE = 'J1743'
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Eligible "Specified" = >=2 Dx dates AND any treatment evidence.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Eligible "Incremental Unspecified" = >=2 Dx dates AND Tivi-only evidence,
-- excluding anyone already in the specified+treatment set.
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible patient list.
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

/* ============================================================================
   2) PROVIDER FILTER ("COHORT 3 LEARNINGS")
   Goal: constrain HCPs to relevant specialties and exclude noise specialties.
   Used to filter Dx/Tx claim NPIs (but still allow NULL NPI claims through).
   ========================================================================== */
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

/* ============================================================================
   3) CLAIMS UNIVERSES (DX + TX) WITH NPI ATTRIBUTION
   - 5Y-ish window (2020-08-01 -> end_date) used for "stats" and "latest"
   - 3Y-ish window (2022-08-01 -> end_date) used for ranking "most-seen"
   ========================================================================== */

-- All diagnosis claims (E761/E763) in the 5Y window, with NPI attribution.
-- Medical uses rendering/referring; pharmacy uses prescriber.
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- Filter to "allowed" NPIs, but keep NULL NPI rows so patient-level dates won't be lost.
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- All treatment claims in the 5Y window, with a unified TX_CODE field:
--   - Tivi NDCs from medical/pharmacy
--   - Infusion/procedure codes from medical (TX_CODE = PROCEDURE_CODE)
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      -- UNION

      -- SELECT DISTINCT
      --     PATIENT_ID,
      --     RENDERING_NPI AS NPI,
      --     SERVICE_DATE AS FILL_DATE,
      --     PROCEDURE_CODE AS TX_CODE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- NEW: ALL-TIME treatment universe (no date restriction) using same tx definition as above.
-- Used ONLY to compute "first_tx_after_diagnosis" without restricting to the 5Y window.
all_tx_claims_alltime AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      -- UNION

      -- SELECT DISTINCT
      --     PATIENT_ID,
      --     RENDERING_NPI AS NPI,
      --     SERVICE_DATE AS FILL_DATE,
      --     PROCEDURE_CODE AS TX_CODE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Normalize Dx + Tx into a single 5Y claim stream (TX_CODE NULL for Dx rows).
-- This enables unified "visit count" and "latest claim" logic.
all_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        CAST(NULL AS STRING) AS TX_CODE
    FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        TX_CODE
    FROM all_tx_claims_5yr
),

-- Combined Dx + Tx claims in the 3Y window for "most-seen HCP" ranking.
all_claims_3yr AS (
    SELECT DISTINCT *
    FROM (
      -- Dx (medical/pharmacy)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      -- Tx (medical/pharmacy/proc)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      -- UNION
      -- SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* ============================================================================
   4) FIRST DX / FIRST TX HCP ATTRIBUTION (5Y WINDOW)
   - "first_dx_hcp": earliest Dx claim NPI per patient (ties broken by NPI)
   - "first_tx_hcp": earliest Tx claim NPI per patient (ties broken by NPI)
   - plus 5Y visit counts + last-visit dates for those attributed HCPs
   ========================================================================== */

first_dx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_dx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_dx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
    FROM first_dx_hcp_ranked
    WHERE rn = 1
),

-- Basic provider dimension for name/specialty lookup.
provider_dim AS (
    SELECT
        npi,
        CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
        primary_specialty
    FROM com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
),

-- For the first Dx-attributed HCP: count all claim dates (Dx+Tx) in 5Y and get last visit.
first_dx_hcp_5yr_stats AS (
    SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
),

first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
    FROM first_tx_hcp_ranked
    WHERE rn = 1
),

-- For the first Tx-attributed HCP: count all claim dates (Dx+Tx) in 5Y and get last visit.
first_tx_hcp_5yr_stats AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

-- For the first Tx-attributed HCP: count treatment claim dates only in 5Y.
first_tx_hcp_5yr_tx_only AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

/* ============================================================================
   5) MOST-SEEN HCP RANKING (TOP 5) USING 3Y ACTIVITY
   - rank by #distinct visit dates in 3Y, then by recency, then by NPI
   - attach 5Y counts and 5Y last visit for those same HCPs
   ========================================================================== */

most_seen_3yr_ranking AS (
    SELECT PATIENT_ID,
           NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI ASC
           ) AS rank
    FROM all_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

most_seen_combined_stats AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        ms3.visit_count_3yr,
        ms3.last_visit_3yr,
        COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
        MAX(ac5.FILL_DATE)          AS last_visit_5yr
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
),

/* ============================================================================
   6) HISTORICAL (ALL-TIME) FIRST DX / FIRST TX DATES (PATIENT LEVEL)
   Goal: get true first Dx date and true first Tx date without windowing.
   ========================================================================== */

historical_first_dx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
    FROM (
        -- Dx specified + unspecified from both medical and pharmacy, no date filters
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_dx
    GROUP BY PATIENT_ID
),

historical_first_tx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
    FROM (
        -- Tx NDCs and procedures, no date filters
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
        --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_tx
    GROUP BY PATIENT_ID
),

/* ============================================================================
   7) LATEST HCP ATTRIBUTION (5Y WINDOW)
   - latest claim HCP (across Dx+Tx): most recent claim date with an NPI
   - latest tx HCP (tx only): most recent tx claim date with an NPI
   Also compute visit counts for those attributed HCPs within the 5Y window.
   ========================================================================== */

all_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      -- Dx (medical/pharmacy)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      -- Tx (medical/pharmacy/proc)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      -- UNION
      -- SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
), 

latest_claim_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
  from all_claims_5yr
),

latest_claim_hcp as (
  select patient_id, npi as latest_claim_hcp_npi, fill_date as latest_claim_date from latest_claim_hcp_ranked where rn = 1
),

latest_claim_hcp_visit_count_5yr as (
  select a.patient_id, a.latest_claim_hcp_npi, count(distinct fill_date) as latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_claim_hcp_npi = b.npi
  group by 1,2
),

latest_claim_hcp_final as (
  select a.patient_id, a.latest_claim_hcp_npi, a.latest_claim_date, b.latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join latest_claim_hcp_visit_count_5yr as b on a.patient_id = b.patient_id
),

-- latest_claim_hcp_ranked AS (
--     SELECT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         ROW_NUMBER() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY FILL_DATE DESC, NPI ASC
--         ) AS rn
--     FROM all_claims_5yr
--     WHERE NPI IS NOT NULL
-- ),
-- latest_claim_hcp AS (
--     SELECT
--         PATIENT_ID,
--         NPI AS latest_claim_hcp_npi,
--         FILL_DATE AS latest_claim_date
--     FROM latest_claim_hcp_ranked
--     WHERE rn = 1
-- ),
-- latest_claim_hcp_visit_count_5yr AS (
--     SELECT
--         lch.PATIENT_ID,
--         lch.latest_claim_hcp_npi,
--         COUNT(DISTINCT ac.FILL_DATE) AS latest_claim_hcp_visit_count_5yr
--     FROM latest_claim_hcp lch
--     LEFT JOIN all_claims_5yr ac
--       ON lch.PATIENT_ID = ac.PATIENT_ID
--      AND lch.latest_claim_hcp_npi = ac.NPI
--     GROUP BY lch.PATIENT_ID, lch.latest_claim_hcp_npi
-- ),

all_tx_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      -- UNION

      -- SELECT DISTINCT
      --     PATIENT_ID,
      --     RENDERING_NPI AS NPI,
      --     SERVICE_DATE AS FILL_DATE,
      --     PROCEDURE_CODE AS TX_CODE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

most_recent_tx_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn 
  from all_tx_claims_5yr
),

most_recent_tx_hcp as (
  select patient_id, npi as latest_treatment_hcp_npi, fill_date as latest_treatment_date
  from most_recent_tx_hcp_ranked where rn = 1
),

latest_treatment_hcp_visit_count as (
  select a.patient_id, a.latest_treatment_hcp_npi, count(distinct fill_date) as latest_treatment_hcp_visit_count_5yr
  from most_recent_tx_hcp as a 
  left join all_tx_claims_5yr as b on a.patient_id = b.patient_id and a.latest_treatment_hcp_npi = b.npi
  group by 1, 2
),

most_recent_tx_hcp_final as (
  select a.patient_id, a.latest_treatment_hcp_npi, a.latest_treatment_date, b.latest_treatment_hcp_visit_count_5yr
  from most_recent_tx_hcp as a
  left join latest_treatment_hcp_visit_count as b on a.patient_id = b.patient_id
),

-- most_recent_tx_hcp_ranked AS (
--     SELECT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         ROW_NUMBER() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY FILL_DATE DESC, NPI ASC
--         ) AS rn
--     FROM all_tx_claims_5yr
--     WHERE NPI IS NOT NULL
-- ),
-- most_recent_tx_hcp AS (
--     SELECT
--         PATIENT_ID,
--         NPI AS latest_treatment_hcp_npi,
--         FILL_DATE AS latest_treatment_date
--     FROM most_recent_tx_hcp_ranked
--     WHERE rn = 1
-- ),
-- latest_treatment_hcp_visit_count AS (
--     SELECT
--         mrt.PATIENT_ID,
--         mrt.latest_treatment_hcp_npi,
--         COUNT(DISTINCT ac.FILL_DATE) AS latest_treatment_hcp_visit_count_5yr
--     FROM most_recent_tx_hcp mrt
--     LEFT JOIN all_claims_5yr ac
--       ON mrt.PATIENT_ID = ac.PATIENT_ID
--      AND mrt.latest_treatment_hcp_npi = ac.NPI
--     GROUP BY mrt.PATIENT_ID, mrt.latest_treatment_hcp_npi
-- ),

/* ============================================================================
   8) PATIENT-LEVEL "LATEST DATE" FALLBACKS (IGNORE NPI)
   Why: if claims exist but all have NULL NPI, HCP-attributed latest_* CTEs go NULL.
        These patient-level dates ensure latest_claim_date/latest_treatment_date are populated.
   ========================================================================== */

-- latest_claim_date_patient AS (
--   SELECT
--     PATIENT_ID,
--     MAX(FILL_DATE) AS latest_claim_date_any
--   FROM all_claims_5yr
--   GROUP BY PATIENT_ID
-- ),
-- latest_treatment_date_patient AS (
--   SELECT
--     PATIENT_ID,
--     MAX(FILL_DATE) AS latest_treatment_date_any
--   FROM all_tx_claims_5yr
--   GROUP BY PATIENT_ID
-- ),

/* ============================================================================
   9) LATEST TX TYPE (WINDOWED TO RECENT TREATMENT PERIOD)
   Goal: classify the latest tx within 2023-08-01..end_date as "Tivi" vs other proc.
   ========================================================================== */

latest_mpsii_treatment_type AS (
  SELECT
    patient_id,
    CASE
      WHEN tx_code IN ('8497600101') THEN 'Tivi - Avalayah'
    END AS latest_mpsii_tx_type
  FROM (
    SELECT
      patient_id,
      fill_date,
      tx_code,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY fill_date DESC, tx_code ASC
      ) AS rn
    FROM all_tx_claims_5yr
    WHERE tx_code IS NOT NULL
      AND fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
  )
  WHERE rn = 1
),

/* ============================================================================
   10) FIRST TX AFTER DIAGNOSIS (ALL-TIME TX, BUT MUST BE AFTER DX)
   Goal: compute earliest treatment date after incidence_date using all_tx_claims_alltime.
   ========================================================================== */

first_tx_after_diagnosis AS (
  SELECT
    tx.patient_id,
    MIN(tx.fill_date) AS first_tx_after_diagnosis
  FROM all_tx_claims_alltime tx
  INNER JOIN historical_first_dx dx
    ON tx.patient_id = dx.patient_id
  WHERE tx.fill_date >= dx.incidence_date
  GROUP BY tx.patient_id
),

/* ============================================================================
   11) Tivi FILL COUNTS (WINDOWED)
   Goal: count distinct treatment dates for Tivi-coded tx between 2023-08-01..end_date.
   ========================================================================== */

Tivi_fills AS (
  SELECT
    patient_id,
    COUNT(DISTINCT fill_date) AS Tivi_fills
  FROM all_tx_claims_5yr
  WHERE fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND tx_code IN ('8497600101')
  GROUP BY patient_id
),

/* ============================================================================
   12) PATIENT DIMENSIONS
   - demographics: pick a single record per patient
   - geography: pick "best current" state using validity logic
   ========================================================================== */

patient_demographics AS (
    SELECT *
    FROM (
        SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
               ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
        FROM com_edp_prd.com_raw.kom_patient_demographics
    )
    WHERE rn = 1
),
patient_geography AS (
    SELECT patient_id, patient_state
    FROM (
        SELECT
            PATIENT_ID,
            patient_state,
            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY
                    CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
                    VALID_TO_DATE DESC
            ) AS rn
        FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
    )
    WHERE rn = 1
),

/* ============================================================================
   13) PIVOT TOP-5 MOST-SEEN HCPs INTO WIDE FORMAT
   Goal: turn rows (patient_id, rank=1..5) into columns to avoid repeated joins.
   ========================================================================== */
most_seen_pivot AS (
    SELECT
        PATIENT_ID,

        MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
        MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
        MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

        MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
        MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
        MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

        MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
        MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
        MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

        MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
        MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
        MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

        MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
        MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
        MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

    FROM most_seen_combined_stats
    GROUP BY PATIENT_ID
)

/* ============================================================================
   FINAL SELECT
   Produces one row per eligible patient with:
   - demographics + geography
   - incidence dates + latest dates (with patient-level fallbacks)
   - attributed HCPs (latest claim, latest tx, first dx, first tx, top 5 most-seen)
   - provider + HCO enrichment (reference_file_pooja_1703)
   - derived metrics and Tivi counts
   ========================================================================== */
SELECT
    ep.PATIENT_ID,
    pd.PATIENT_YOB,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.patient_state,

    -- Historical First Dates (ALL-TIME)
    hfdx.incidence_date,
    hftx.first_incidence_treatment_date,

    -- Latest claim date: use HCP-attributed latest if available else patient-level fallback
    -- COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,
    lch.latest_claim_date AS latest_claim_date,

    -- Latest claim HCP attribution (only when NPI exists on that latest claim)
    lch.latest_claim_hcp_npi,
    pdlch.provider_name     AS latest_claim_hcp_name,
    pdlch.primary_specialty AS latest_claim_hcp_specialty,
    COALESCE(lch.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

    -- Map latest-claim HCP -> HCO via reference crosswalk
    -- ref1.hco_npi  AS latest_claim_hcp_hco_npi,
    ref1.hco_name AS latest_claim_hcp_hco_name,

    -- Latest treatment date: use HCP-attributed latest if available else patient-level fallback
    mrt.latest_treatment_date AS latest_treatment_date,

    -- Latest tx type (windowed to 2023-08-01..end_date)
    lmt.latest_mpsii_tx_type,

    -- First treatment after diagnosis (ALL-TIME tx, constrained to >= incidence_date)
    fta.first_tx_after_diagnosis,

    -- Derived timing: dx -> first tx (months)
    ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
      AS time_dx_to_first_tx_in_months,

    -- Derived timing: tx period (months) = first tx after dx -> latest tx (patient-level)
    ROUND(MONTHS_BETWEEN(mrt.latest_treatment_date, fta.first_tx_after_diagnosis), 0) AS treatment_period_months,

    -- Tivi fills (windowed, Tivi-only codes)
    COALESCE(ef.Tivi_fills, 0) AS Tivi_fills,

    -- Latest treatment HCP attribution (only when NPI exists on that latest tx claim)
    mrt.latest_treatment_hcp_npi,
    pdtch.provider_name     AS latest_treatment_hcp_name,
    pdtch.primary_specialty AS latest_treatment_hcp_specialty,
    COALESCE(mrt.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,

    -- Map latest-tx HCP -> HCO via reference crosswalk
    -- ref2.hco_npi  AS latest_treatment_hcp_hco_npi,
    ref2.hco_name AS latest_treatment_hcp_hco_name,

    -- First Dx HCP (5Y stats)
    fdh.first_dx_hcp AS first_dx_hcp_5yr,
    COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
    fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

    -- First Tx HCP (5Y stats)
    fth.first_tx_hcp AS first_tx_hcp_5yr,
    COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
    COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
    fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

    -- Top 5 most-seen HCPs (ranked by 3Y, with 5Y stats)
    msp.most_seen_hcp1_3yr_ranked,
    COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
    msp.most_seen_hcp1_last_visit_5yr,

    msp.most_seen_hcp2_3yr_ranked,
    COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
    msp.most_seen_hcp2_last_visit_5yr,

    msp.most_seen_hcp3_3yr_ranked,
    COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
    msp.most_seen_hcp3_last_visit_5yr,

    msp.most_seen_hcp4_3yr_ranked,
    COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
    msp.most_seen_hcp4_last_visit_5yr,

    msp.most_seen_hcp5_3yr_ranked,
    COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
    msp.most_seen_hcp5_last_visit_5yr

FROM eligible_patients ep
LEFT JOIN patient_demographics pd
    ON ep.PATIENT_ID = pd.PATIENT_ID
LEFT JOIN patient_geography pg
    ON ep.PATIENT_ID = pg.PATIENT_ID

-- Patient-level ALL-TIME incidence dates
LEFT JOIN historical_first_dx hfdx
    ON ep.PATIENT_ID = hfdx.PATIENT_ID
LEFT JOIN historical_first_tx hftx
    ON ep.PATIENT_ID = hftx.PATIENT_ID

-- First tx after dx (ALL-TIME)
LEFT JOIN first_tx_after_diagnosis fta
    ON ep.PATIENT_ID = fta.PATIENT_ID

-- Tivi fills (windowed)
LEFT JOIN Tivi_fills ef
    ON ep.PATIENT_ID = ef.PATIENT_ID

-- Patient-level latest date fallbacks (ignore NPI)
-- LEFT JOIN latest_claim_date_patient lcd
--     ON ep.PATIENT_ID = lcd.PATIENT_ID
-- LEFT JOIN latest_treatment_date_patient ltd
--     ON ep.PATIENT_ID = ltd.PATIENT_ID

-- Latest tx type (windowed)
LEFT JOIN latest_mpsii_treatment_type lmt
    ON ep.PATIENT_ID = lmt.PATIENT_ID

-- Latest claim HCP + enrichment (provider + HCO)
LEFT JOIN latest_claim_hcp_final lch
    ON ep.PATIENT_ID = lch.PATIENT_ID
-- LEFT JOIN latest_claim_hcp_visit_count_5yr lchvc
--     ON ep.PATIENT_ID = lchvc.PATIENT_ID
--    AND lch.latest_claim_hcp_npi = lchvc.latest_claim_hcp_npi
LEFT JOIN provider_dim pdlch
    ON lch.latest_claim_hcp_npi = pdlch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file_pooja_1703 ref1
    ON lch.latest_claim_hcp_npi = ref1.hcp_npi

-- Latest tx HCP + enrichment (provider + HCO)
LEFT JOIN most_recent_tx_hcp_final mrt
    ON ep.PATIENT_ID = mrt.PATIENT_ID
-- LEFT JOIN latest_treatment_hcp_visit_count lthvc
--     ON ep.PATIENT_ID = lthvc.PATIENT_ID
--    AND mrt.latest_treatment_hcp_npi = lthvc.latest_treatment_hcp_npi
LEFT JOIN provider_dim pdtch
    ON mrt.latest_treatment_hcp_npi = pdtch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file_pooja_1703 ref2
    ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

-- First Dx/Tx HCP attribution + stats
LEFT JOIN first_dx_hcp fdh
    ON ep.PATIENT_ID = fdh.PATIENT_ID
LEFT JOIN first_dx_hcp_5yr_stats fdhs
    ON ep.PATIENT_ID = fdhs.PATIENT_ID
LEFT JOIN first_tx_hcp fth
    ON ep.PATIENT_ID = fth.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_stats fths
    ON ep.PATIENT_ID = fths.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
    ON ep.PATIENT_ID = fthtx.PATIENT_ID

-- Pivoted top-5 most-seen HCPs
LEFT JOIN most_seen_pivot msp
    ON ep.PATIENT_ID = msp.PATIENT_ID

ORDER BY ep.PATIENT_ID;

-- Materialize the temp view into the persistent base table.
CREATE OR Replace TEMPORARY VIEW patient360_base AS
SELECT DISTINCT * FROM patient_hcp_visit_summary;


In [0]:
%sql
-- =============================================================================
-- mpsii_tx_claims (Temp View)
--
-- What this view is:
--   A curated “treatment claims” (Tx) universe for an MPS II eligible patient cohort.
--   It outputs Tx events (medical NDC, pharmacy NDC, and procedure administrations)
--   with an attributed HCP NPI when available, within the refresh window.
--
-- Output grain:
--   One row per (patient_id, npi-attribution, fill_date) treatment event.
--   Note: NPI can be NULL (kept intentionally).
--
-- Key inputs:
--   - com_edp_prd.com_raw.kom_medical_events
--   - com_edp_prd.com_raw.kom_pharmacy_events
--   - com_raw.kom_providers  (for provider filtering)
--
-- Key logic:
--   1) Build eligible_patients using Dx evidence (E761/E763) + treatment evidence.
--   2) Define provider inclusion list (cohort_3_learnings) based on specialties.
--   3) Pull treatment events in refresh window and filter to included providers (or NULL NPI).
--
-- Parameters:
--   ${end_date} should be supplied by the runtime (Databricks widget/job parameter).
-- =============================================================================

CREATE OR REPLACE TEMP VIEW mpsii_tx_claims AS
WITH
-- ============================================================================
-- 1) Dx evidence for cohort building (Specified vs Unspecified)
--    These CTEs create Dx event streams used ONLY to count distinct Dx dates.
--    Window used here: 2020-08-01 → ${end_date}
-- ============================================================================

-- Specified Dx events:
--   Pull E761 diagnosis occurrences from:
--     (a) medical_events where DIAGNOSIS_CODES contains E761, using SERVICE_DATE
--     (b) pharmacy_events where DIAGNOSIS_CODE = E761 and transaction is PAID, using FILL_DATE
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Specified Dx patients:
--   Keep patients with at least 2 distinct Dx dates (>=2 distinct fill_date values).
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Unspecified Dx events:
--   Pull E763 diagnosis occurrences from medical + paid pharmacy, same as above.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Unspecified Dx patients:
--   Keep patients with at least 2 distinct Dx dates.
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- ============================================================================
-- 2) Treatment evidence for cohort building (refresh window only)
--    These CTEs do NOT output the final Tx universe; they’re used to confirm
--    that a patient has qualifying treatment evidence in the refresh window.
--    Window used here: 2023-08-01 → ${end_date}
-- ============================================================================

-- Treatment evidence (broad):
--   Patient qualifies if they have ANY of:
--     - Tivi NDCs in medical (NDC11) within refresh window
--     - Tivi NDCs in pharmacy within refresh window with PAID result
--     - Any procedure/admin codes in the provided procedure list within refresh window
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Treatment evidence (Tivi-only):
--   Narrower evidence set for incremental unspecified cohort:
--     - Tivi NDCs (medical/pharmacy) in refresh window
--     - J1743 procedure in refresh window
MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE = 'J1743'
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- ============================================================================
-- 3) Build eligible patient cohort
--    - Specified: >=2 E761 Dx dates AND ANY treatment evidence in refresh window
--    - Incremental unspecified: >=2 E763 Dx dates AND Tivi-only evidence
--      AND not already in specified+treatment cohort
-- ============================================================================

-- Specified cohort:
--   Patients who meet the “>=2 specified Dx dates” requirement
--   AND have at least one qualifying treatment event in refresh window.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Incremental unspecified cohort:
--   Patients who meet the “>=2 unspecified Dx dates” requirement
--   AND have Tivi-only evidence in refresh window
--   AND are not in the specified+treatment cohort (avoids double-counting).
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible cohort:
--   Union of specified+treatment cohort and incremental unspecified cohort.
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ============================================================================
-- 4) Provider inclusion filter (cohort_3_learnings)
--    Goal: Restrict attributed NPIs to INDIVIDUAL providers that meet specialty rules.
--    This filter will be applied AFTER pulling treatment events.
--    Note: rows with NULL NPI are retained to preserve treatment dates even without attribution.
-- ============================================================================

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      -- Exclude a set of primary specialties considered out-of-scope/noise
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant',
        'Anesthesiology',
        'Dentist',
        'Dietitian, Registered',
        'Emergency Medical Technician, Basic',
        'Emergency Medicine',
        'General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered',
        'Obstetrics & Gynecology',
        'Pathology',
        'Radiology',
        'Urology'
      )
      -- OR explicitly include certain secondary specialties that are relevant
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry',
        'Psychiatry',
        'Adolescent Medicine',
        'Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine',
        'Nutrition, Pediatric',
        'Oncology, Pediatrics',
        'Pediatric Cardiology',
        'Pediatric Critical Care Medicine',
        'Pediatric Dermatology',
        'Pediatric Emergency Medicine',
        'Pediatric Endocrinology',
        'Pediatric Gastroenterology',
        'Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases',
        'Pediatric Nephrology',
        'Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery',
        'Pediatric Otolaryngology',
        'Pediatric Pulmonology',
        'Pediatric Radiology',
        'Pediatric Rehabilitation Medicine',
        'Pediatric Rheumatology',
        'Pediatric Surgery',
        'Pediatrics',
        'Clinical Biochemical Genetics',
        'Clinical Genetics (M.D.)',
        'Clinical Molecular Genetics',
        'Ph.D. Medical Genetics',
        'Neurodevelopmental Disabilities',
        'Neurology',
        'Neurology with Special Qualifications in Child Neurology',
        'Neuroradiology'
      )
    )
),

-- ============================================================================
-- 5) Treatment claims universe returned by the view (refresh/“2y” window)
--    Pull Tx events for eligible patients during 2023-08-01 → ${end_date}.
--    Sources and attribution rules:
--      A) Medical NDC events: NPI = COALESCE(rendering_npi, referring_npi), date = SERVICE_DATE
--      B) Pharmacy NDC events: NPI = prescriber_npi, date = FILL_DATE, PAID only
--      C) Procedure events:    NPI = rendering_npi, date = SERVICE_DATE
--    Then apply provider filter:
--      - keep if NPI in cohort_3_learnings OR NPI is NULL
-- ============================================================================

all_tx_claims_2yr AS (
    SELECT DISTINCT *
    FROM (
      -- Medical: Tivi NDC events with attributed HCP (rendering/referring)
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      -- Pharmacy: Tivi NDC fills with attributed prescriber (paid only)
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      -- UNION

      -- -- Medical: procedure/admin events with attributed renderer
      -- SELECT DISTINCT
      --   PATIENT_ID,
      --   RENDERING_NPI AS NPI,
      --   SERVICE_DATE AS FILL_DATE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- Provider specialty filter: keep included providers OR keep NULL NPI rows
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
)

-- Final output: all treatment events in refresh window for the eligible cohort
SELECT * FROM all_tx_claims_2yr;


In [0]:
%sql
-- =============================================================================
-- most_recently_treated_hcp (Temp View)
--
-- What this view is:
--   Patient-level attribution of the “most recently treating HCP” within the
--   REFRESH treatment window (2023-08-01 → ${end_date}), plus HCP enrichment and
--   visit context metrics computed over a broader (5y-ish) claims universe.
--
-- Output grain:
--   One row per patient (only patients with an attributable treatment NPI in the
--   refresh window will appear, because latest_treating_hcp filters npi IS NOT NULL).
--
-- Key concepts:
--   - Selection window ("most recently treated"): tx_claims (refresh window)
--   - Context window ("visits / last seen"): all_claims (currently 2020-08-01 → ${end_date})
--   - Provider filter (cohort_3_learnings): keeps included INDIVIDUAL NPIs; retains NULL NPI rows
--     in claims universes, but final attribution requires non-null NPI.
--
-- Parameters:
--   ${end_date} must be provided by the runtime.
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
WITH
-- ============================================================================
-- 1) Treatment claims: refresh window source
--    This CTE simply points to the already-built tx universe from mpsii_tx_claims:
--      - eligible cohort already applied
--      - window already applied (2023-08-01 → ${end_date})
--      - provider filter already applied (allowed NPIs + NULL NPIs)
-- ============================================================================
tx_claims AS (
    SELECT DISTINCT *
    FROM mpsii_tx_claims
),

-- ============================================================================
-- 2) Re-derive eligible_patients cohort (duplicated here)
--    This block repeats the cohort logic so that the subsequent 5y Dx/Tx universes
--    (all_dx_claims_5yr / all_tx_claims_5yr) can be built inside this view.
--    NOTE: This duplication is intentional in the current script; no logic is changed.
-- ============================================================================

-- Specified Dx event stream (E761) within 2020-08-01 → ${end_date}
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Specified Dx patients: require ≥2 distinct Dx dates
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Unspecified Dx event stream (E763) within 2020-08-01 → ${end_date}
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Unspecified Dx patients: require ≥2 distinct Dx dates
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Qualifying treatment evidence (broad) in refresh window, used to ensure “active treatment”
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Tivi-only evidence in refresh window (used only for incremental unspecified cohort)
MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE = 'J1743'
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Specified eligible: ≥2 specified Dx dates AND any qualifying treatment in refresh window
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Incremental unspecified eligible: ≥2 unspecified Dx dates AND Tivi-only tx in refresh window,
-- excluding already eligible specified+treatment patients
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible cohort used downstream for 5y Dx/Tx universes
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ============================================================================
-- 3) Provider filter (cohort_3_learnings)
--    Defines an allowed set of INDIVIDUAL NPIs based on specialty inclusion rules.
--    Applied to all_dx_claims_5yr / all_tx_claims_5yr, with NULL NPIs retained.
-- ============================================================================
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

-- ============================================================================
-- 4) Build 5y-ish claims universes for visit counts / last-visit context
--    These are bounded by 2020-08-01 → ${end_date} and filtered to eligible_patients.
--    Provider filter is applied (allowed NPIs + NULL NPIs retained).
-- ============================================================================

-- Dx claims in 5y-ish window (E761/E763)
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Tx claims in 5y-ish window (Tivi NDCs + procedure list)
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      -- UNION
      -- SELECT DISTINCT
      --   PATIENT_ID,
      --   RENDERING_NPI AS NPI,
      --   SERVICE_DATE AS FILL_DATE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Combined claims universe used only for:
--   - counting distinct visit dates per patient↔HCP
--   - computing last observed visit date per patient↔HCP
all_claims AS (
    SELECT * FROM all_dx_claims_5yr
    UNION
    SELECT * FROM all_tx_claims_5yr
),

-- ============================================================================
-- 5) Select the most recently treating HCP (refresh window)
--    Uses tx_claims (2023-08-01 → ${end_date}) to pick 1 NPI per patient:
--      - Prefer rows with non-null NPI
--      - Then pick the latest fill_date
--      - Tie-break by npi DESC (deterministic tie-breaker)
--    Final output from this CTE requires npi IS NOT NULL (attributable HCP).
-- ============================================================================
latest_treating_hcp AS (
    SELECT patient_id, npi
    FROM (
        SELECT
            patient_id,
            npi,
            fill_date,
            ROW_NUMBER() OVER (
                PARTITION BY patient_id
                ORDER BY
                    CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,  -- prioritize attributed claims
                    fill_date DESC,                               -- most recent date wins
                    npi DESC                                      -- deterministic tie-break
            ) AS rn
        FROM tx_claims
    ) t
    WHERE rn = 1
      AND npi IS NOT NULL
),

-- ============================================================================
-- 6) Compute visit metrics for the selected patient↔HCP pair (5y-ish universe)
--    These metrics are NOT limited to the refresh window; they use all_claims.
-- ============================================================================

-- Count of distinct visit dates for each patient↔HCP across all_claims
visit_counts AS (
    SELECT
        patient_id,
        npi,
        COUNT(DISTINCT fill_date) AS visit_counts
    FROM all_claims
    WHERE npi IS NOT NULL
    GROUP BY patient_id, npi
),

-- Last observed visit date for each selected patient↔HCP across all_claims
last_visit_date AS (
    SELECT
        lth.patient_id,
        lth.npi,
        MAX(ac.fill_date) AS last_visit_date
    FROM latest_treating_hcp lth
    LEFT JOIN all_claims ac
      ON lth.patient_id = ac.patient_id
     AND lth.npi = ac.npi
    GROUP BY lth.patient_id, lth.npi
),

-- Attach visit metrics to the most recently treated HCP per patient
latest_treating_hcp_with_visits AS (
    SELECT
        a.patient_id,
        a.npi AS most_recently_treated_hcp,
        b.visit_counts AS no_of_visits,
        c.last_visit_date AS last_visit_date_5yr
    FROM latest_treating_hcp AS a
    LEFT JOIN visit_counts AS b
        ON a.patient_id = b.patient_id
       AND a.npi        = b.npi
    LEFT JOIN last_visit_date AS c
        ON a.patient_id = c.patient_id
       AND a.npi        = c.npi
),

-- ============================================================================
-- 7) Enrichment: provider name/specialty + HCO + territory/region mapping
--    - provider info from kom_providers (INDIVIDUAL)
--    - HCO / territory / region crosswalk from reference_file_pooja_1703
--      with additional territory_id/region_id derived via zip_to_territory_mapping
-- ============================================================================
hcp_with_other_info AS (
    SELECT
        a.*,
        CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
        b.PRIMARY_SPECIALTY AS hcp_specialty,
        -- c.hco_npi,
        c.hco_name,
        c.territory_id,
        c.territory,
        c.region_id,
        c.region
    FROM latest_treating_hcp_with_visits AS a

    -- Provider name and specialty enrichment (limit to INDIVIDUAL provider rows)
    LEFT JOIN com_edp_prd.com_raw.kom_providers AS b
        ON a.most_recently_treated_hcp = b.NPI
       AND b.PROVIDER_TYPE = 'INDIVIDUAL'

    -- Crosswalk HCP -> HCO and attach territory/region metadata.
    -- Inner derived tables map territory_name/region_name to numeric IDs.
    LEFT JOIN (
      SELECT
        * EXCEPT (hcp_primary_specialty),
        hcp_primary_specialty AS hcp_specialty
      FROM (
        SELECT
          a.*,
          b.territory_id,
          c.region_id
        FROM cmpa_insights_internal_schema.reference_file_pooja_1703 AS a
        LEFT JOIN (
          SELECT DISTINCT territory_id, territory_name
          FROM cmpa_insights_internal_schema.zip_to_territory_mapping
        ) AS b
          ON a.territory = b.territory_name
        LEFT JOIN (
          SELECT DISTINCT region_id, region_name
          FROM cmpa_insights_internal_schema.zip_to_territory_mapping
        ) AS c
          ON a.region = c.region_name
      )
    ) AS c
        ON a.most_recently_treated_hcp = c.hcp_npi
)

-- ============================================================================
-- FINAL SELECT
--   Renames fields to make explicit:
--     - selection window: “_2yr” (refresh window)
--     - context metrics: “_5yr” (computed from all_claims 2020-08-01 → ${end_date})
-- ============================================================================
SELECT
    patient_id,
    most_recently_treated_hcp AS most_recently_treated_hcp_2yr,
    hcp_name AS most_recently_treated_hcp_name_2yr,

    -- Visit metrics are computed across all_claims (currently 2020-08-01 → ${end_date})
    no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,
    last_visit_date_5yr AS most_recent_tx_hcp_2yr_last_visit_5yr,

    hcp_specialty AS most_recently_treated_hcp_specialty_2yr,
    -- hco_npi AS most_recently_treated_hcp_hco_npi,
    hco_name AS most_recently_treated_hcp_hco_name,
    territory_id AS most_recently_treated_hcp_territory_id_2yr,
    territory AS most_recently_treated_hcp_territory_2yr,
    region_id AS most_recently_treated_hcp_region_id_2yr,
    region AS most_recently_treated_hcp_region_2yr
FROM hcp_with_other_info;


In [0]:
%sql
-- =============================================================================
-- Primary HCP Assignment (Dx + Tx Claims)
--
-- Goal
--   Assign ONE “primary HCP” (NPI) per eligible MPS II patient by ranking HCPs using:
--     Tier 1) Specialty priority
--     Tier 2) Total distinct visit dates (Dx + Tx combined)
--     Tier 3) Most recent visit date
--     Tier 4) NPI tiebreaker
--
-- Date windows used in this script
--   • Dx claim extraction (“5y” universe): 2020-08-01 → ${end_date}
--   • Tx claim extraction (“5y” universe): 2020-08-01 → ${end_date}
--   • Tx eligibility window (“2y/refresh”): 2023-08-01 → ${end_date}
--
-- NPI attribution (aligned to GTM file)
--   • Medical NDC claims:      COALESCE(RENDERING_NPI, REFERRING_NPI)
--   • Medical procedure claims:RENDERING_NPI
--   • Pharmacy claims:         PRESCRIBER_NPI
--
-- Eligibility overview (eligible_patients)
--   • Specified cohort:
--       - ≥2 distinct E761 Dx dates (medical or paid pharmacy) in Dx window
--       - AND any Tx in the 2y/refresh window
--   • Incremental unspecified cohort:
--       - ≥2 distinct E763 Dx dates (medical or paid pharmacy) in Dx window
--       - AND Tivi-coded Tx in the 2y/refresh window (NDCs or J1743)
--       - Excludes patients already in specified cohort
-- =============================================================================


-- =============================================================================
-- STEP 1: DIAGNOSIS CLAIMS (Dx universe for visit counting)
--   • Captures E761/E763 diagnosis evidence from medical + paid pharmacy events
--   • Window: 2020-08-01 → ${end_date}
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical events Dx (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

-- Pharmacy events Dx (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (Tx universe for visit counting)
--   • Captures Tivi-coded treatment from medical NDC, medical procedures, and paid pharmacy NDC
--   • Window: 2020-08-01 → ${end_date}
--   • Includes CODE field to retain NDC/procedure provenance
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical events Tx via NDC (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('8497600101')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- -- Medical events Tx via procedures (NPI = rendering_npi only)
-- SELECT DISTINCT
--     PATIENT_ID,
--     RENDERING_NPI AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     PROCEDURE_CODE AS CODE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
--                          'S9357', 'S9379', '38206', '38230', '38232',
--                          '38240', '38241', '38242', '38243', '38250')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

-- Pharmacy events Tx via NDC (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('8497600101')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 3: Tx CLAIMS IN ELIGIBILITY WINDOW (2y/refresh)
--   • Subset of all_tx_claims restricted to 2023-08-01 → ${end_date}
--   • Used ONLY to determine cohort eligibility (not for visit counting tiers)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
--   Builds eligible_patients using Dx evidence (≥2 dates) + Tx evidence in 2y window.
-- =============================================================================

-- 4A) Specified Dx requirement: ≥2 distinct E761 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4B) Specified cohort: specified Dx + any Tx in 2y/refresh window
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

-- 4C) Incremental Dx requirement: ≥2 distinct E763 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4D) Tivi-coded Tx requirement for incremental eligibility (2y/refresh window)
CREATE OR REPLACE TEMPORARY VIEW Tivi_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('8497600101');

-- 4E) Incremental cohort: incremental Dx + Tivi-coded tx in 2y window + exclude specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN Tivi_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- 4F) Final eligible cohort
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- Provider inclusion list (INDIVIDUAL NPIs only)
--   • Used to restrict HCPs considered for primary assignment
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
SELECT DISTINCT npi
FROM com_raw.kom_providers
WHERE provider_type = 'INDIVIDUAL'
  AND (
    PRIMARY_SPECIALTY NOT IN (
      'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
      'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
      'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
      'Radiology','Urology'
    )
    OR SECONDARY_SPECIALTY IN (
      'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
      'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
      'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
      'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
      'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
      'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
      'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
      'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
      'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
      'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
    )
  );


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
--   • Combines Dx + Tx events (visit dates) for eligible patients only
--   • Filters to included INDIVIDUAL NPIs (cohort_3_learnings)
--   • NOTE: As written, this excludes NULL NPI rows (because of the IN filter)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
SELECT *
FROM (
  -- Dx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  -- Tx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)
WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


-- =============================================================================
-- STEP 6: PRIMARY HCP ASSIGNMENT (4-tier ranking)
--   Tier 1: Specialty priority bucket (lower = better)
--   Tier 2: Total distinct visit dates (Dx + Tx)
--   Tier 3: Most recent visit date
--   Tier 4: NPI tiebreaker
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty bucket (reporting label)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Tier 1: specialty priority (lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: visit counts (Dx + Tx combined)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference-only breakdowns (not used in ranking)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: most recent visit date
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;

-- Optional materialization:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
-- SELECT * FROM primary_hcp;


In [0]:
%sql
-- =============================================================================
-- STEP 7: Materialize Primary HCP table with HCP identity + HCO/territory metadata
--
-- What this step does:
--   Takes the existing primary_hcp assignment (already computed upstream)
--   and persists it as a physical table, while enriching it with:
--     - HCP full name (from kom_providers)
--     - HCO affiliation + territory/region attributes (from reference_file_pooja_1703)
--     - territory_id / region_id (derived via zip_to_territory_mapping lookups)
--
-- What this step does NOT do:
--   - It does not re-rank or change the primary HCP selection logic.
--   - It does not filter the cohort; it simply enriches and stores primary_hcp rows.
--
-- Output:
--   com_edp_prd.cmpa_insights_internal_schema.primary_hcp
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW primary_hcp_base AS
SELECT
    ph.*,  -- retain all columns produced by the upstream primary_hcp view/table

    -- -------------------------------------------------------------------------
    -- HCP display name enrichment
    --   Concatenate first + last name from provider dimension; COALESCE protects
    --   against nulls so string concat doesn't produce NULL.
    -- -------------------------------------------------------------------------
    COALESCE(p.FIRST_NAME, '') || ' ' || COALESCE(p.LAST_NAME, '') AS primary_hcp_name_2yr,

    -- -------------------------------------------------------------------------
    -- HCO affiliation enrichment (via crosswalk)
    --   Map HCP NPI -> HCO NPI / HCO name using reference_file_pooja_1703.
    -- -------------------------------------------------------------------------
    -- ref.HCO_NPI  AS primary_hcp_hco_npi_2yr,
    ref.HCO_NAME AS primary_hcp_hco_name_2yr,

    -- -------------------------------------------------------------------------
    -- Territory / region enrichment
    --   Pull both the human-readable names and numeric IDs (territory_id, region_id).
    -- -------------------------------------------------------------------------
    ref.territory_id AS primary_hcp_territory_id_2yr,
    ref.TERRITORY    AS primary_hcp_territory_2yr,
    ref.region_id    AS primary_hcp_region_id_2yr,
    ref.region       AS primary_hcp_region_2yr

FROM primary_hcp ph

-- -----------------------------------------------------------------------------
-- Join #1: Provider dimension (name enrichment)
--   Join on the attributed primary HCP NPI to retrieve FIRST_NAME / LAST_NAME.
-- -----------------------------------------------------------------------------
LEFT JOIN com_edp_prd.com_raw.kom_providers p
    ON ph.PRIMARY_HCP_NPI = p.NPI

-- -----------------------------------------------------------------------------
-- Join #2: Reference enrichment (HCO + territory + region)
--   Build a reference subquery that:
--     1) Starts from reference_file_pooja_1703 (HCP ↔ HCO + territory/region names)
--     2) Adds territory_id by mapping territory name -> territory_id via zip_to_territory_mapping
--     3) Adds region_id by mapping region name -> region_id via zip_to_territory_mapping
--     4) Renames hcp_primary_specialty to hcp_specialty (not used in final select here,
--        but retained in the ref dataset)
--   Finally join ref to primary_hcp using HCP_NPI.
-- -----------------------------------------------------------------------------
LEFT JOIN (
    SELECT
      * EXCEPT (hcp_primary_specialty),
      hcp_primary_specialty AS hcp_specialty
    FROM (
      SELECT
        a.*,
        b.territory_id,
        c.region_id
      FROM cmpa_insights_internal_schema.reference_file_pooja_1703 AS a

      -- Map territory name -> territory_id (distinct pairs)
      LEFT JOIN (
        SELECT DISTINCT territory_id, territory_name
        FROM cmpa_insights_internal_schema.zip_to_territory_mapping
      ) AS b
        ON a.territory = b.territory_name

      -- Map region name -> region_id (distinct pairs)
      LEFT JOIN (
        SELECT DISTINCT region_id, region_name
        FROM cmpa_insights_internal_schema.zip_to_territory_mapping
      ) AS c
        ON a.region = c.region_name
    )
) ref
    ON ph.PRIMARY_HCP_NPI = ref.HCP_NPI;


In [0]:
%sql
-- =============================================================================
-- patient360_master (Final patient-level master table)
--
-- What this step does:
--   Materializes a single, “wide” patient-level table by combining three upstream
--   datasets into one row per patient:
--     A) patient360_base              -> core demographics + claim-derived milestones + HCP features
--     B) most_recently_treated_hcp    -> most recent treating HCP in refresh window + visit context metrics
--     C) primary_hcp                  -> primary HCP assignment + HCO/territory/region enrichment
--
-- Output:
--   com_edp_prd.cmpa_insights_internal_schema.patient360_master
--
-- Join strategy:
--   - Start from patient360_base (a) as the backbone (all patients retained)
--   - LEFT JOIN most_recently_treated_hcp (b) on patient_id to add recent-treatment attribution fields
--   - LEFT JOIN primary_hcp (c) on patient_id to add primary HCP and territory enrichment fields
--   - SELECT DISTINCT used to dedupe in case joins introduce multiplicity (e.g., enrichment tables
--     or upstream views have >1 row per patient)
--
-- Column handling:
--   - b.* EXCEPT(patient_id) and c.* EXCEPT(patient_id) avoid duplicating patient_id columns
--     from the joined datasets.
--   - Window suffixes (_2yr/_3yr/_5yr) indicate derivation windows from upstream logic.
--
-- Parameters:
--   None directly here (all parameterized logic occurs upstream), but this depends on upstream
--   tables/views that may be parameterized by ${end_date}.
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW patient360_master AS
select PATIENT_ID as tivi_patient_id, PATIENT_YOB as tivi_patient_yob, PATIENT_AGE as tivi_patient_age, PATIENT_GENDER as tivi_patient_gender, patient_state as tivi_patient_state, incidence_date as tivi_incidence_date, first_incidence_treatment_date as tivi_first_incidence_treatment_date, latest_claim_date as tivi_latest_claim_date, latest_claim_hcp_npi as tivi_latest_claim_hcp_npi, latest_claim_hcp_name as tivi_latest_claim_hcp_name, latest_claim_hcp_specialty as tivi_latest_claim_hcp_specialty, latest_claim_hcp_visit_count as tivi_latest_claim_hcp_visit_count, latest_claim_hcp_hco_name as tivi_latest_claim_hcp_hco_name, latest_treatment_date as tivi_latest_treatment_date, latest_mpsii_tx_type as tivi_latest_mpsii_tx_type, first_tx_after_diagnosis as tivi_first_tx_after_diagnosis, time_dx_to_first_tx_in_months as tivi_time_dx_to_first_tx_in_months, treatment_period_months as tivi_treatment_period_months, Tivi_fills as tivi_fills, latest_treatment_hcp_npi as tivi_latest_treatment_hcp_npi, latest_treatment_hcp_name as tivi_latest_treatment_hcp_name, latest_treatment_hcp_specialty as tivi_latest_treatment_hcp_specialty, latest_treatment_hcp_visit_count as tivi_latest_treatment_hcp_visit_count, latest_treatment_hcp_hco_name as tivi_latest_treatment_hcp_hco_name, first_dx_hcp_5yr as tivi_first_dx_hcp_5yr, first_dx_all_visit_count_5yr as tivi_first_dx_all_visit_count_5yr, first_dx_last_visit_5yr as tivi_first_dx_last_visit_5yr, first_tx_hcp_5yr as tivi_first_tx_hcp_5yr, first_tx_all_visit_count_5yr as tivi_first_tx_all_visit_count_5yr, first_tx_treatment_visit_count_5yr as tivi_first_tx_treatment_visit_count_5yr, first_tx_last_visit_5yr as tivi_first_tx_last_visit_5yr, most_seen_hcp1_3yr_ranked as tivi_most_seen_hcp1_3yr_ranked, most_seen_hcp1_visit_count_5yr as tivi_most_seen_hcp1_visit_count_5yr, most_seen_hcp1_last_visit_5yr as tivi_most_seen_hcp1_last_visit_5yr, most_seen_hcp2_3yr_ranked as tivi_most_seen_hcp2_3yr_ranked, most_seen_hcp2_visit_count_5yr as tivi_most_seen_hcp2_visit_count_5yr, most_seen_hcp2_last_visit_5yr as tivi_most_seen_hcp2_last_visit_5yr, most_seen_hcp3_3yr_ranked as tivi_most_seen_hcp3_3yr_ranked, most_seen_hcp3_visit_count_5yr as tivi_most_seen_hcp3_visit_count_5yr, most_seen_hcp3_last_visit_5yr as tivi_most_seen_hcp3_last_visit_5yr, most_seen_hcp4_3yr_ranked as tivi_most_seen_hcp4_3yr_ranked, most_seen_hcp4_visit_count_5yr as tivi_most_seen_hcp4_visit_count_5yr, most_seen_hcp4_last_visit_5yr as tivi_most_seen_hcp4_last_visit_5yr, most_seen_hcp5_3yr_ranked as tivi_most_seen_hcp5_3yr_ranked, most_seen_hcp5_visit_count_5yr as tivi_most_seen_hcp5_visit_count_5yr, most_seen_hcp5_last_visit_5yr as tivi_most_seen_hcp5_last_visit_5yr, most_recently_treated_hcp_2yr as tivi_most_recently_treated_hcp_2yr, most_recently_treated_hcp_name_2yr as tivi_most_recently_treated_hcp_name_2yr, most_recently_treated_hcp_2yr_no_of_visits_5yr as tivi_most_recently_treated_hcp_2yr_no_of_visits_5yr, most_recent_tx_hcp_2yr_last_visit_5yr as tivi_most_recent_tx_hcp_2yr_last_visit_5yr, most_recently_treated_hcp_specialty_2yr as tivi_most_recently_treated_hcp_specialty_2yr, most_recently_treated_hcp_hco_name as tivi_most_recently_treated_hcp_hco_name, most_recently_treated_hcp_territory_id_2yr as tivi_most_recently_treated_hcp_territory_id_2yr, most_recently_treated_hcp_territory_2yr as tivi_most_recently_treated_hcp_territory_2yr, most_recently_treated_hcp_region_id_2yr as tivi_most_recently_treated_hcp_region_id_2yr, most_recently_treated_hcp_region_2yr as tivi_most_recently_treated_hcp_region_2yr, PRIMARY_HCP_NPI as tivi_primary_hcp_npi, PRIMARY_HCP_SPECIALTY as tivi_primary_hcp_specialty, SPECIALTY_PRIORITY as tivi_specialty_priority, NO_OF_VISITS as tivi_no_of_visits, DX_VISITS as tivi_dx_visits, TX_VISITS as tivi_tx_visits, MOST_RECENT_VISIT as tivi_most_recent_visit, HCP_RANK as tivi_hcp_rank


from (SELECT DISTINCT
    -- -------------------------------------------------------------------------
    -- Patient identity + demographics (from patient360_base)
    -- -------------------------------------------------------------------------
    a.PATIENT_ID,
    a.PATIENT_YOB,
    a.PATIENT_AGE,
    a.PATIENT_GENDER,
    a.patient_state,

    -- -------------------------------------------------------------------------
    -- Historical milestones (patient-level; timeframe-agnostic where defined upstream)
    --   - incidence_date: earliest observed Dx date across history (as defined in base)
    --   - first_incidence_treatment_date: earliest observed treatment date across history (as defined in base)
    -- -------------------------------------------------------------------------
    a.incidence_date,
    a.first_incidence_treatment_date,

    -- -------------------------------------------------------------------------
    -- Latest claim (patient-level)
    --   - latest_claim_date may fall back to a patient-level date if NPI attribution is missing upstream
    -- -------------------------------------------------------------------------
    a.latest_claim_date,

    -- -------------------------------------------------------------------------
    -- Latest claim attributed HCP + visit context (from patient360_base)
    --   Note: these fields are populated only if the latest claim had an attributable NPI upstream
    -- -------------------------------------------------------------------------
    a.latest_claim_hcp_npi,
    a.latest_claim_hcp_name,
    a.latest_claim_hcp_specialty,
    a.latest_claim_hcp_visit_count,

    -- Latest claim attributed HCO (via reference enrichment in base)
    -- a.latest_claim_hcp_hco_npi,
    a.latest_claim_hcp_hco_name,

    -- -------------------------------------------------------------------------
    -- Latest treatment (patient-level) + derived treatment type
    --   - latest_treatment_date may fall back to a patient-level date if NPI attribution is missing upstream
    --   - latest_mpsii_tx_type typically derived within the refresh window upstream
    -- -------------------------------------------------------------------------
    a.latest_treatment_date,
    a.latest_mpsii_tx_type,

    -- -------------------------------------------------------------------------
    -- First Tx after Dx + timelines (from patient360_base)
    --   - first_tx_after_diagnosis is computed upstream (all-time tx constrained to >= incidence_date)
    --   - timelines are derived from incidence_date / first_tx_after_diagnosis / latest_treatment_date
    -- -------------------------------------------------------------------------
    a.first_tx_after_diagnosis,
    a.time_dx_to_first_tx_in_months,
    a.treatment_period_months,

    -- -------------------------------------------------------------------------
    -- Treatment activity proxy (from patient360_base)
    --   - Tivi_fills: distinct tx dates in refresh window as defined upstream
    -- -------------------------------------------------------------------------
    a.Tivi_fills,

    -- -------------------------------------------------------------------------
    -- Latest treatment attributed HCP + visit context (from patient360_base)
    -- -------------------------------------------------------------------------
    a.latest_treatment_hcp_npi,
    a.latest_treatment_hcp_name,
    a.latest_treatment_hcp_specialty,
    a.latest_treatment_hcp_visit_count,

    -- Latest treatment attributed HCO (via reference enrichment in base)
    -- a.latest_treatment_hcp_hco_npi,
    a.latest_treatment_hcp_hco_name,

    -- -------------------------------------------------------------------------
    -- First Dx / Tx attributed HCP metrics (5y-ish universe; from patient360_base)
    -- -------------------------------------------------------------------------
    a.first_dx_hcp_5yr,
    a.first_dx_all_visit_count_5yr,
    a.first_dx_last_visit_5yr,
    a.first_tx_hcp_5yr,
    a.first_tx_all_visit_count_5yr,
    a.first_tx_treatment_visit_count_5yr,
    a.first_tx_last_visit_5yr,

    -- -------------------------------------------------------------------------
    -- Top-5 most-seen HCPs (ranked by 3y; includes 5y counts + last-visit; from patient360_base)
    -- -------------------------------------------------------------------------
    a.most_seen_hcp1_3yr_ranked,
    a.most_seen_hcp1_visit_count_5yr,
    a.most_seen_hcp1_last_visit_5yr,
    a.most_seen_hcp2_3yr_ranked,
    a.most_seen_hcp2_visit_count_5yr,
    a.most_seen_hcp2_last_visit_5yr,
    a.most_seen_hcp3_3yr_ranked,
    a.most_seen_hcp3_visit_count_5yr,
    a.most_seen_hcp3_last_visit_5yr,
    a.most_seen_hcp4_3yr_ranked,
    a.most_seen_hcp4_visit_count_5yr,
    a.most_seen_hcp4_last_visit_5yr,
    a.most_seen_hcp5_3yr_ranked,
    a.most_seen_hcp5_visit_count_5yr,
    a.most_seen_hcp5_last_visit_5yr,

    -- -------------------------------------------------------------------------
    -- Add-on enrichment block #1: most_recently_treated_hcp
    --   Pulls in:
    --     - most_recently_treated_hcp_2yr and its attributes (name/specialty/HCO/territory/region)
    --     - visit context metrics computed over the broader claims universe in that view
    --   EXCEPT(patient_id) avoids duplicating the patient id column.
    -- -------------------------------------------------------------------------
    b.* EXCEPT (patient_id),

    -- -------------------------------------------------------------------------
    -- Add-on enrichment block #2: primary_hcp
    --   Pulls in:
    --     - primary HCP attribution + name/HCO/territory/region fields
    --   EXCEPT(patient_id) avoids duplicating the patient id column.
    -- -------------------------------------------------------------------------
    c.* EXCEPT (patient_id)

FROM patient360_base AS a

-- Left join retains all patients from patient360_base even if no match in most_recently_treated_hcp
LEFT JOIN most_recently_treated_hcp AS b
    ON a.PATIENT_ID = b.patient_id

-- Left join retains all patients from patient360_base even if no match in primary_hcp
LEFT JOIN primary_hcp AS c
    ON a.PATIENT_ID = c.patient_id)


In [0]:
%sql
select * from patient360_master;